In [ ]:
%load_ext autoreload
%autoreload 2

### Import and run model

In [1]:
# Forward pass with an untrained model.
from cs336_basics.model import TransformerLM
import torch

# device=torch.device("cuda")
device = torch.device("mps")
model = TransformerLM(
  vocab_size=10000,
  context_length=256,
  num_layers=8,
  d_model=768,
  num_heads=16,
  d_ff=1344,
  rope_theta=10000.0,
  device=device,
  dtype=torch.float32,
)

input = torch.randint(0, 10000, (1, 256), device=device)
output = model(input)
print(output.shape)

torch.Size([1, 256, 10000])


In [ ]:
# Generate text using a trained model.
from cs336_basics.generation import generate_text
from cs336_basics.tokenizer import BPETokenizer
from cs336_basics.model import TransformerLM
from cs336_basics.io import ROOT_PATH, load_checkpoint

tok = BPETokenizer.load(ROOT_PATH / "tokenizer/tinystories_train_10000.pt")

model, _, _ = load_checkpoint(
  ROOT_PATH / "model/TinyStories/volcanic-dream-4/checkpoint_39999.pt",
  model_class=TransformerLM,
)
text = generate_text(
  model=model,
  tokenizer=tok,
  input_text= "Costanza",
  max_new_tokens=100,
  temperature=1.0,
  top_p=0.0,
)
print(model._init_args)
text

### Models to benchmark

In [ ]:
from benchmark import MODELS, instantiate_model, model_size_mb

for name, model_config in MODELS.items():
  model = instantiate_model(model_config, 256)
  total_size_mb = model_size_mb(model)
  print(f"{name} model size: {total_size_mb:.2f} MB")

# small model size: 491.42 MB
# medium model size: 1615.82 MB
# large model size: 3700.26 MB
# xl model size: 7625.66 MB
# 2.7B model size: 12998.45 MB

### Basic Benchmark

In [ ]:
%%bash
cd ..
uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --no-synchronize \
    --no-measure-also-backward

uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --synchronize \
    --no-measure-also-backward

uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --synchronize \
    --measure-also-backward

In [ ]:
# warmup_steps = 0. Notice the high std.
# --- small model (491.42 MB) ---
# without_sync_without_backward: 0.0892s ± 0.1655s
# with_sync_without_backward: 0.0303s ± 0.0024s
# with_sync_with_backward: 0.1095s ± 0.1056s
# --- medium model (1615.82 MB) ---
# without_sync_without_backward: 0.0818s ± 0.0434s
# with_sync_without_backward: 0.0659s ± 0.0001s
# with_sync_with_backward: 0.1975s ± 0.0004s
# --- large model (3700.26 MB) ---
# without_sync_without_backward: 0.1123s ± 0.0449s
# with_sync_without_backward: 0.1399s ± 0.0002s
# with_sync_with_backward: 0.4234s ± 0.0001s
# --- xl model (7625.66 MB) ---
# without_sync_without_backward: 0.2115s ± 0.1135s
# with_sync_without_backward: 0.2845s ± 0.0001s
# with_sync_with_backward: 0.8637s ± 0.0622s
# --- 2.7B model (12998.45 MB) ---
# without_sync_without_backward: 0.1527s ± 0.0012s
# with_sync_without_backward: 0.4181s ± 0.0001s
# with_sync_with_backward: 1.2662s ± 0.0234s

# warmup_steps = 5. Much lower std than before.
# --- small model (491.42 MB) ---
# without_sync_without_backward: 0.0312s ± 0.0002s
# with_sync_without_backward: 0.0340s ± 0.0032s
# with_sync_with_backward: 0.0767s ± 0.0008s
# --- medium model (1615.82 MB) ---
# without_sync_without_backward: 0.0633s ± 0.0013s
# with_sync_without_backward: 0.0670s ± 0.0014s
# with_sync_with_backward: 0.1997s ± 0.0014s
# --- large model (3700.26 MB) ---
# without_sync_without_backward: 0.0975s ± 0.0019s
# with_sync_without_backward: 0.1409s ± 0.0001s
# with_sync_with_backward: 0.4241s ± 0.0001s
# --- xl model (7625.66 MB) ---
# without_sync_without_backward: 0.1711s ± 0.0007s
# with_sync_without_backward: 0.2852s ± 0.0001s
# with_sync_with_backward: 0.8440s ± 0.0004s
# --- 2.7B model (12998.45 MB) ---
# without_sync_without_backward: 0.1548s ± 0.0024s
# with_sync_without_backward: 0.4184s ± 0.0001s
# with_sync_with_backward: 1.2589s ± 0.0012s